# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# 21 · Frozen-model relationship signal audit

**No training, no target arrays, no additional validation scoring.**

The same 32 training plays are inspected with the two final Round 8 models. Prediction-change RMS measures sensitivity, not forecast error. A branch can influence predictions without helping generalization, and destructive interventions can create unrealistic input combinations.

In [ ]:
from pathlib import Path
import json, sys, subprocess
import plotly.io as pio
KIT = Path('/home/sagemaker-user/nfl_feature_round9')
OUT = Path('/home/sagemaker-user/nfl-feature-round9-results')
PY = Path('/home/sagemaker-user/nfl-player-trajectory/.venv/bin/python')
if not KIT.is_dir() or not PY.is_file():
    raise FileNotFoundError('Open the existing NFL space and extract Round 9 first.')
sys.path.insert(0, str(KIT))
import visuals
pio.renderers.default = 'plotly_mimetype'
def run(stage):
    process = subprocess.Popen([str(PY), str(KIT/'run_round.py'), stage],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='')
        code = process.wait()
    except KeyboardInterrupt:
        process.send_signal(2)
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
    if code:
        raise RuntimeError(f'{stage} stopped ({code}). Preserve inputs and export the report; do not train or change settings.')
def show(fig, name):
    visuals.save(fig, OUT, name).show()


## Inspect both frozen models

Require `signal_audit_complete`. The runner reuses the original CPU PyTorch lock offline. No runtime installation, optimizer, or training command is present. It saves per-play diagnostic receipts for resumption.

In [ ]:
run('audit')
s = json.loads((OUT/'signal_summary.json').read_text())
print(json.dumps({k:s[k] for k in ['status','training_plays','training_rows','new_scientific_models','new_optimizer_steps','new_official_rmse']}, indent=2))

## What information differs, and what reaches predictions?

Compare original history/terminal inputs, then the change in predictions under fixed weights. The gradient plot uses two signed projections to avoid simple output-sum cancellation; it is a local estimate, not a feature ranking. Weight movement is descriptive only.

In [ ]:
show(visuals.pair_information(OUT), 'pair_input_difference')
show(visuals.prediction_changes(OUT), 'frozen_prediction_sensitivity')
show(visuals.channel_gradients(OUT), 'pair_channel_gradient')
show(visuals.parameter_movement(OUT), 'parameter_movement')

## Fresh-process replay

Require `signal_audit_replay_exact` and `new_optimizer_steps: 0`. All diagnostic records and the aggregate are reconstructed exactly. Old sources, inputs and final weight blobs must remain byte-identical.

In [ ]:
run('replay')
print(json.dumps(json.loads((OUT/'replay.json').read_text()), indent=2))

## Export, then stop

Return this small aggregate ZIP. Do not launch another model or tune based on the stress-test charts. The next scientific experiment depends on whether input variation is missing, the branch responds weakly, or meaningful sensitivity exists without generalization.

In [ ]:
run('report')
print(OUT/'nfl_feature_round9_report.zip')